# Met Eyes Experiments

# Process Data

## Install

In [ ]:
from google.colab import userdata
PAT_XYZ = userdata.get("PAT_XYZ")

In [ ]:
!pip install mediapipe
!pip install ultralytics
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!wget https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite
!wget https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_full_range/float16/latest/blaze_face_full_range.tflite
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py
!git clone https://{PAT_XYZ}@github.com/acervos-digitais/met-faces-data.git data

## Setup

In [ ]:
import json
import numpy as np
import requests

from os import listdir, makedirs, path
from PIL import Image as PImage, ImageDraw as PImageDraw, ImageEnhance as PImageEnhance
from time import sleep

from mediapipe import Image as mpImage, ImageFormat as mpImageFormat
from mediapipe.tasks.python.core.base_options import BaseOptions as mpBaseOptions
from mediapipe.tasks.python.vision import FaceDetector as mpFaceDetector, FaceLandmarker as mpFaceLandmarker
from mediapipe.tasks.python.vision import FaceDetectorOptions as mpFaceDetectorOptions
from mediapipe.tasks.python.vision import FaceLandmarkerOptions as mpFaceLandmarkerOptions
from mediapipe.tasks.python.vision import RunningMode as mpRunningMode

from ultralytics import YOLO

from utils import export_combined_jsons
from utils import pxs_to_pcts, pcts_to_sqs, pct_to_px

DATA_DIR = "./lehman-data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

IMG_FACES_DIR = f"{IMG_DIR}/faces"

JSON_FACES_DIR = f"{JSON_DIR}/faces"
JSON_LANDMARK_DIR = f"{JSON_DIR}/landmarks"

## Analyze Faces

### Face Detection

In [ ]:
makedirs(JSON_FACES_DIR, exist_ok=True)

In [ ]:
img_ids = sorted(int(fn.replace(".jpg", "")) for fn in listdir(f"{IMG_DIR}/900") if fn.endswith(".jpg"))

with open(f"{JSON_DIR}/objects.json", "r") as ifp:
  obj_data = json.load(ifp)["objects"]
  id2obj = { obj["objectID"] : obj for obj in obj_data }

### Detect Faces ([YOLO11](https://huggingface.co/AdamCodd/YOLOv11n-face-detection))

In [ ]:
yolo_model_path = hf_hub_download(repo_id="AdamCodd/YOLOv11n-face-detection", filename="model.pt")
face_detector = YOLO(yolo_model_path)

In [ ]:
for cnt,oid in enumerate(img_ids):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(img_ids)}")

  obj = id2obj[oid]

  face_json_path = f"{JSON_FACES_DIR}/{oid}.json"
  if path.isfile(face_json_path):
    with open(face_json_path, "r") as ifp:
      obj = json.load(ifp)

  if "faces" in obj and "yolo" in obj["faces"]:
    continue

  img = PImage.open(f"{IMG_DIR}/900/{oid}.jpg")
  iw,ih = img.size
  nh = 256
  nw = int(nh * iw // ih)
  nimg = img.resize((nw, nh))

  faces = face_detector.predict(nimg, verbose=False, device="cuda")
  if len(faces) < 1 or len(faces[0]) < 1:
    continue

  faces_xyxyn = faces[0].boxes.xyxyn.cpu().numpy().astype(np.float64)
  faces_xyxyn_sq = pcts_to_sqs(faces_xyxyn, iw, ih)

  if "faces" not in obj:
    obj["faces"] = {}

  obj["faces"]["yolo"] = {
    "count": len(faces_xyxyn),
    "xyxyn": faces_xyxyn.round(4).tolist(),
    "xyxyn_sq": faces_xyxyn_sq.round(4).tolist(),
  }

  with open(face_json_path, "w") as ofp:
    json.dump(obj, ofp)

### Detect Faces (Zero-Shot)

In [ ]:
from huggingface_hub import hf_hub_download
from torch import no_grad, Tensor
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, pipeline

In [ ]:
MODEL_NAME = "IDEA-Research/grounding-dino-base"

zs_processor = AutoProcessor.from_pretrained(MODEL_NAME)
zs_model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_NAME).to("cuda")

labels = ["face"]

In [ ]:
for cnt,oid in enumerate(img_ids):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(img_ids)}")

  obj = id2obj[oid]

  face_json_path = f"{JSON_FACES_DIR}/{oid}.json"
  if path.isfile(face_json_path):
    with open(face_json_path, "r") as ifp:
      obj = json.load(ifp)

  if "faces" in obj and "dino" in obj["faces"]:
    continue

  img = PImage.open(f"{IMG_DIR}/900/{oid}.jpg")
  iw,ih = img.size

  with no_grad():
    input = zs_processor(text=labels, images=img, return_tensors="pt").to("cuda")
    output = zs_model(**input)

  res = zs_processor.post_process_grounded_object_detection(outputs=output, target_sizes=[Tensor([ih, iw])], threshold=0.33)

  if len(res[0]["boxes"]) < 1:
    continue

  faces_xyxyn = pxs_to_pcts(res[0]["boxes"].cpu(), iw, ih, xyxy=True).astype(np.float64)
  faces_xyxyn_sq = pcts_to_sqs(faces_xyxyn, iw, ih)

  if "faces" not in obj:
    obj["faces"] = {}

  obj["faces"]["dino"] = {
    "count": len(res[0]["boxes"]),
    "xyxyn": faces_xyxyn.round(4).tolist(),
    "xyxyn_sq": faces_xyxyn_sq.round(4).tolist(),
  }

  with open(face_json_path, "w") as ofp:
    json.dump(obj, ofp)

In [ ]:
export_combined_jsons(JSON_FACES_DIR, JSON_DIR, "faces", ["faces"])

### Detect Faces (Media-Pipe) (NOT VERY GOOD, actually)

https://ai.google.dev/edge/mediapipe/solutions/vision/face_detector/python

In [ ]:
makedirs(JSON_FACES_DIR, exist_ok=True)

In [ ]:
img_ids = sorted(int(fn.replace(".jpg", "")) for fn in listdir(f"{IMG_DIR}/900") if fn.endswith(".jpg"))

with open(f"{JSON_DIR}/objects.json", "r") as ifp:
  obj_data = json.load(ifp)["objects"]
  id2obj = { obj["objectID"] : obj for obj in obj_data }

In [ ]:
face_model_path = "./blaze_face_full_range.tflite"

face_options = mpFaceDetectorOptions(
  base_options=mpBaseOptions(model_asset_path=face_model_path),
  running_mode=mpRunningMode.IMAGE,
  min_detection_confidence=0.5
)

detector = mpFaceDetector.create_from_options(face_options)

In [ ]:
for cnt,oid in enumerate(img_ids):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(img_ids)}")

  obj = id2obj[oid]

  face_json_path = f"{JSON_FACES_DIR}/{oid}.json"
  if path.isfile(face_json_path):
    with open(face_json_path, "r") as ifp:
      obj = json.load(ifp)

  if "faces" in obj and "mp_faces" in obj["faces"]:
    continue

  pimg = PImage.open(f"{IMG_DIR}/900/{oid}.jpg")
  iw,ih = pimg.size
  mp_image = mpImage(image_format=mpImageFormat.SRGB, data=np.array(pimg))
  face_detector_result = detector.detect(mp_image)

  if len(face_detector_result.detections) < 1:
    continue

  faces_xyxy = [[f.bounding_box.origin_x,
                 f.bounding_box.origin_y,
                 f.bounding_box.origin_x + f.bounding_box.width,
                 f.bounding_box.origin_y + f.bounding_box.height]
                for f in face_detector_result.detections]
  faces_xyxyn = pxs_to_pcts(faces_xyxy, iw, ih, xyxy=True).astype(np.float64)
  faces_xyxyn_sq = pcts_to_sqs(faces_xyxyn, iw, ih)

  if "faces" not in obj:
    obj["faces"] = {}

  obj["faces"]["mp_faces"] = {
    "count": len(face_detector_result.detections),
    "xyxyn": faces_xyxyn.round(4).tolist(),
    "xyxyn_sq": faces_xyxyn_sq.round(4).tolist(),
  }

  with open(face_json_path, "w") as ofp:
    json.dump(obj, ofp)

### Crop Faces

In [ ]:
makedirs(IMG_FACES_DIR, exist_ok=True)

In [ ]:
face_img_ids = set(int(f.split("_")[0]) for f in listdir(IMG_FACES_DIR) if f.endswith("jpg"))

print(len(face_img_ids))

with open(f"{JSON_DIR}/faces.json", "r") as ifp:
  obj_data = json.load(ifp)["faces"]

In [ ]:
for ocnt,obj in enumerate(obj_data):
  if ocnt % 20 == 0:
    print(f"{ocnt} / {len(obj_data)}")

  oid = obj["objectID"]

  if oid in face_img_ids:
    continue

  if not ("faces" in obj and "yolo" in obj["faces"]):
    continue

  img_response = requests.get(obj["primaryImage"], stream=True)
  img = PImage.open(img_response.raw)
  iw,ih = img.size

  for fcnt,box in enumerate(obj["faces"]["yolo"]["xyxyn_sq"]):
    x0,y0,x1,y1 = pct_to_px(box, iw, ih)

    face_img_cnt_str = f"000{fcnt}"[-3:]
    face_img_fname = f"{oid}_{face_img_cnt_str}"
    face_img = img.crop((x0,y0,x1,y1)).convert("RGB")
    face_img.save(f"{IMG_DIR}/faces/{face_img_fname}.jpg")

  face_img_ids.add(oid)
  sleep(0.333)

### Face Landmarks: Eyes and Gaze (Media Pipe)

https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/python


Landmarks:

<img src="imgs/face-landmarks.jpg" height=300 />
<img src="imgs/face-landmarks-eyes.jpg" height=300 />

In [ ]:
makedirs(JSON_LANDMARK_DIR, exist_ok=True)

In [ ]:
landmarker_model_path = "./face_landmarker.task"

landmarker_options = mpFaceLandmarkerOptions(
  base_options=mpBaseOptions(model_asset_path=landmarker_model_path),
  running_mode=mpRunningMode.IMAGE
)

landmarker = mpFaceLandmarker.create_from_options(landmarker_options)

### Initial Run

In [ ]:
with open(f"{JSON_DIR}/faces.json", "r") as ifp:
  obj_data = json.load(ifp)["faces"]

In [ ]:
for ocnt,obj in enumerate(obj_data):
  if ocnt % 20 == 0:
    print(f"{ocnt} / {len(obj_data)}")

  oid = obj["objectID"]

  if not ("faces" in obj and "yolo" in obj["faces"]):
    continue

  mp_results = {
    "count": 0,
    "landmarks": [],
  }

  for fcnt,fbox in enumerate(obj["faces"]["yolo"]["xyxyn_sq"]):
    fx0,fy0,fx1,fy1 = fbox
    fw, fh = (fx1 - fx0), (fy1 - fy0)

    face_img_cnt_str = f"000{fcnt}"[-3:]
    face_img_fname = f"{oid}_{face_img_cnt_str}"

    pimg = PImage.open(f"{IMG_FACES_DIR}/{face_img_fname}.jpg").resize((512, 512))
    mp_image = mpImage(image_format=mpImageFormat.SRGB, data=np.array(pimg))
    landmarks = landmarker.detect(mp_image)

    face_landmarks = np.array([]).astype(np.float64)

    if len(landmarks.face_landmarks) > 0:
      mp_results["count"] += 1
      face_landmarks = np.array([[fx0 + lm.x * fw, fy0 + lm.y * fh] for lm in landmarks.face_landmarks[0]]).astype(np.float64)
    else:
      pimg = PImage.open(f"{IMG_FACES_DIR}/{face_img_fname}.jpg").resize((64, 64))
      mp_image = mpImage(image_format=mpImageFormat.SRGB, data=np.array(pimg))
      landmarks = landmarker.detect(mp_image)

      if len(landmarks.face_landmarks) > 0:
        mp_results["count"] += 1
        face_landmarks = np.array([[fx0 + lm.x * fw, fy0 + lm.y * fh] for lm in landmarks.face_landmarks[0]]).astype(np.float64)

    mp_results["landmarks"].append(face_landmarks.round(4).tolist())

  if "mp" not in obj["faces"]:
    obj["faces"]["mp"] = {}

  for k,v in mp_results.items():
    obj["faces"]["mp"][k] = v

  with open(f"{JSON_LANDMARK_DIR}/{oid}.json", "w") as ofp:
    json.dump(obj, ofp)

In [ ]:
export_combined_jsons(JSON_LANDMARK_DIR, JSON_DIR, "landmarks", ["faces"])

### Draw some landmarks

In [ ]:
obj = obj_data[10]
oid = obj["objectID"]

print(oid)
print(obj["faces"]["mp"]["count"], "/", obj["faces"]["yolo"]["count"])

dimg = PImage.open(f"./lehman-data/image/500/{oid}.jpg")
diw,dih = dimg.size
draw = PImageDraw.Draw(dimg)

r = 2
for mask in obj["faces"]["mp"]["landmarks"]:
  for xyn in mask:
    x,y = xyn[0] * diw, xyn[1] * dih
    draw.ellipse((x - r, y - r, x + r, y + r), fill=(220, 0, 0))

display(dimg)

### Revisit Missed Landmarks

In [ ]:
with open(f"{JSON_DIR}/landmarks.json", "r") as ifp:
  obj_data = json.load(ifp)["landmarks"]

In [ ]:
for ocnt,obj in enumerate(obj_data):
  if ocnt % 20 == 0:
    print(f"{ocnt} / {len(obj_data)}")

  oid = obj["objectID"]

  if not ("faces" in obj and "yolo" in obj["faces"]):
    continue

  if "faces" in obj and "mp" in obj["faces"]:
    if obj["faces"]["mp"]["count"] == obj["faces"]["yolo"]["count"]:
      continue

  mp_results = {
    "count": 0,
    "landmarks": [],
  }
  previous_landmarks = obj["faces"].get("mp", {}).get("landmarks", [])

  for fcnt,fbox in enumerate(obj["faces"]["yolo"]["xyxyn_sq"]):
    if fcnt < len(previous_landmarks):
      if len(previous_landmarks[fcnt]) > 0:
        mp_results["landmarks"].append(previous_landmarks[fcnt])
        mp_results["count"] += 1
        continue

    fx0,fy0,fx1,fy1 = fbox
    fw, fh = (fx1 - fx0), (fy1 - fy0)

    face_img_cnt_str = f"000{fcnt}"[-3:]
    face_img_fname = f"{oid}_{face_img_cnt_str}"

    pimg = PImage.open(f"{IMG_FACES_DIR}/{face_img_fname}.jpg").resize((64, 64))
    pimg = PImageEnhance.Contrast(pimg).enhance(4.5).convert("L").convert("RGB")
    mpimg = mpImage(image_format=mpImageFormat.SRGB, data=np.array(pimg))
    landmarks = landmarker.detect(mpimg)

    face_landmarks = np.array([]).astype(np.float64)

    if len(landmarks.face_landmarks) > 0:
      mp_results["count"] += 1
      face_landmarks = np.array([[fx0 + lm.x * fw, fy0 + lm.y * fh] for lm in landmarks.face_landmarks[0]]).astype(np.float64)
    else:
      pimg = PImage.open(f"{IMG_FACES_DIR}/{face_img_fname}.jpg").resize((64, 64))
      pimg = PImageEnhance.Contrast(pimg).enhance(5.5).convert("L").convert("RGB")
      mpimg = mpImage(image_format=mpImageFormat.SRGB, data=np.array(pimg))
      landmarks = landmarker.detect(mpimg)

      if len(landmarks.face_landmarks) > 0:
        mp_results["count"] += 1
        face_landmarks = np.array([[fx0 + lm.x * fw, fy0 + lm.y * fh] for lm in landmarks.face_landmarks[0]]).astype(np.float64)
      else:
        pimg = PImage.open(f"{IMG_FACES_DIR}/{face_img_fname}.jpg").resize((64, 64))
        pimg = PImageEnhance.Brightness(pimg).enhance(1.666).convert("L").convert("RGB")
        mpimg = mpImage(image_format=mpImageFormat.SRGB, data=np.array(pimg))
        landmarks = landmarker.detect(mpimg)

        if len(landmarks.face_landmarks) > 0:
          mp_results["count"] += 1
          face_landmarks = np.array([[fx0 + lm.x * fw, fy0 + lm.y * fh] for lm in landmarks.face_landmarks[0]]).astype(np.float64)

    if len(landmarks.face_landmarks) > 0:
      dimg = pimg.copy()
      diw, dih = dimg.size
      draw = PImageDraw.Draw(dimg)
      face_landmarks_local = np.array([[lm.x, lm.y] for lm in landmarks.face_landmarks[0]]).astype(np.float64)
      r = 1
      for xyn in face_landmarks_local[::10]:
        x,y = xyn[0] * diw, xyn[1] * dih
        draw.ellipse((x - r, y - r, x + r, y + r), fill=(220, 0, 0))
      display(dimg.resize((100,100)))

    mp_results["landmarks"].append(face_landmarks.round(4).tolist())

  if "mp" not in obj["faces"]:
    obj["faces"]["mp"] = {}

  for k,v in mp_results.items():
    obj["faces"]["mp"][k] = v

  with open(f"{JSON_LANDMARK_DIR}/{oid}.json", "w") as ofp:
    json.dump(obj, ofp)

In [ ]:
export_combined_jsons(JSON_LANDMARK_DIR, JSON_DIR, "landmarks", ["faces"])

In [ ]:
# TODO: Landmark Detection
  # TODO: https://huggingface.co/kartiknarayan/facexformer
  # TODO: https://huggingface.co/qualcomm/Facial-Landmark-Detection